#

In [1]:
# ==============================================================================
# CELDA 1: CONFIGURACIÓN GENERAL Y CARGA DE LA INFRAESTRUCTURA ANALÍTICA
# ==============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier

# Forzar al entorno de Jupyter a posicionarse en la raíz del proyecto
raiz_proyecto = Path().resolve().parent
os.chdir(raiz_proyecto)

# Asegurar que la raíz esté en el path para futuras importaciones de 'src/'
if str(raiz_proyecto) not in sys.path:
    sys.path.insert(0, str(raiz_proyecto))

print(f"✅ Directorio de trabajo establecido en la raíz: {os.getcwd()}")

# Cargar el dataset procesado limpio
df = pd.read_csv('data/raw/incidencia_delictiva.csv', encoding='latin-1')

# Mapear la variable temporal cronológica
orden_meses = {
    'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
    'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12
}
df['mes_num'] = df['mes'].map(orden_meses)

print(f"Dataset cargado exitosamente. Dimensiones: {df.shape[0]} registros.")

✅ Directorio de trabajo establecido en la raíz: C:\Users\Admin\Downloads\Escuela\mineria\Proyecto_Mineria\Proyecto_Final_Mineria_Datos
Dataset cargado exitosamente. Dimensiones: 413952 registros.

In [2]:
# ==============================================================================
# CELDA 2: ENTRENAMIENTO DEL CLASIFICADOR Y GENERACIÓN DEL ARTEFACTO JOBLIB
# ==============================================================================
print("=== INICIANDO EXPORTACIÓN DEL MODELO SUPERVISADO DE DEFENSA ===")

# 1. Separar variables explicativas (Features) y variable objetivo (Target)
features = ['anio', 'mes_num', 'clave_ent', 'incidencia_delictiva']
target = 'bien_juridico_afectado'

X = df[features]
y = df[target]

# 2. Instanciar el estimador optimizado con balanceo geométrico de pesos
SEED = 42
modelo_produccion = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

# 3. Ajustar el modelo con la totalidad de la muestra histórica nacional
print("Entrenando el clasificador Random Forest de producción...")
modelo_produccion.fit(X, y)

# 4. Asegurar la existencia del directorio destino y guardar físicamente el archivo
Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_produccion, "models/modelo_final.joblib")

print("\n=======================================================")
print("✅ ARTEFACTO GENERADO: models/modelo_final.joblib")
print("=======================================================")

=== INICIANDO EXPORTACIÓN DEL MODELO SUPERVISADO DE DEFENSA ===
Entrenando el clasificador Random Forest de producción...

✅ ARTEFACTO GENERADO: models/modelo_final.joblib

In [3]:
# ==============================================================================
# CELDA 3: SIMULACIÓN DE INFERENCIA EN TIEMPO REAL (PRUEBA DE REUTILIZACIÓN)
# ==============================================================================
print("=== SIMULACIÓN DE REUTILIZACIÓN DEL COMPONENTE SERIALIZADO ===")

# Cargar el archivo binario simulando un entorno externo desconectado de la etapa de entrenamiento
modelo_cargado = joblib.load("models/modelo_final.joblib")
print("✅ Componente binario cargado en memoria exitosamente.\n")

# Crear un caso sintético de prueba: Diciembre, Ciudad de México (Clave 9), 450 denuncias
caso_prueba = pd.DataFrame({
    'anio': [2025],
    'mes_num': [12],
    'clave_ent': [9],
    'incidencia_delictiva': [450]
})

# Ejecutar inferencia predictiva al vuelo (baja latencia)
prediccion = modelo_cargado.predict(caso_prueba)[0]
probabilidades = modelo_cargado.predict_proba(caso_prueba)[0]
confianza = probabilidades.max()

print("-------------------------------------------------------")
print("📊 RESULTADO DE INFERENCIA SINTÉTICA:")
print("-------------------------------------------------------")
print(f"🔮 Categoría delictiva predicha : {prediccion}")
print(f"🎯 Grado de certidumbre métrica : {confianza * 100:.2f}%")
print("-------------------------------------------------------")

=== SIMULACIÓN DE REUTILIZACIÓN DEL COMPONENTE SERIALIZADO ===
✅ Componente binario cargado en memoria exitosamente.

-------------------------------------------------------
📊 RESULTADO DE INFERENCIA SINTÉTICA:
-------------------------------------------------------
🔮 Categoría delictiva predicha : Otros bienes jurÃ­dicos afectados (del fuero comÃºn)
🎯 Grado de certidumbre métrica : 47.65%
-------------------------------------------------------

In [ ]:
> ### 📝 Interpretación Técnica de la Reutilización de Componentes Estables
> * **Qué se observa:** Se comprueba analíticamente que tras congelar y guardar los pesos matemáticos del estimador *Random Forest* en un archivo físico `.joblib`, el objeto puede ser cargado instantáneamente en una sesión de Python limpia. Al alimentarlo con un vector numérico sintético de contexto regional (`anio=2025`, `mes_num=12`, `clave_ent=9`, `incidencia_delictiva=450`), el modelo genera la etiqueta de clasificación y su vector de probabilidades sin necesidad de ejecutar ni una sola línea de ajuste (*fit*).
> * **Implicación para el análisis y despliegue:** Esto demuestra el completo desacoplamiento entre el ciclo de vida de desarrollo de software y la puesta en producción. El componente binario encapsula de forma opaca toda la ingeniería de variables y penalizaciones calculadas. Garantiza una latencia mínima de respuesta frente a nuevas peticiones en el script interactivo `predict.py`, eliminando la sobrecarga computacional de almacenamiento en memoria y permitiendo su portabilidad directa a arquitecturas de API rest o interfaces en vivo.